In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error

In [ ]:
import os
import sys
from pathlib import Path

for _p in [Path.cwd(), *Path.cwd().parents]:
    if (_p / ".git").exists():
        ROOT = _p
        break
os.chdir(ROOT)
print(f"Project root: {ROOT}")

sys.path.insert(0, str(ROOT))
from src.encoding import encode_categoricals

In [ ]:
# ── 1. Load a split ──────────────────────────────────────────────────────────
# Start with "random"; swap for "topology" or "metal" to test generalization
STRATEGY = "random"
SPLITS = ROOT / "data/splits"

train = pd.read_parquet(SPLITS / f"split_{STRATEGY}_train_merged.parquet")
val   = pd.read_parquet(SPLITS / f"split_{STRATEGY}_val_merged.parquet")
test  = pd.read_parquet(SPLITS / f"split_{STRATEGY}_test_merged.parquet")

In [ ]:
# ── 2. Choose a feature set ──────────────────────────────────────────────────
feat_info = json.loads((ROOT / "data/merged/feature_selection.json").read_text())

# Options: "baseline", "geo_decorr", "geo_all",
#          "rac_decorr", "rac_all", "combined_decorr", "combined_all"
FEATURE_SET = "combined_decorr"

feature_cols = feat_info["feature_sets"][FEATURE_SET]
target_cols  = feat_info["target_cols"]   # 5 CO2 pressure points

In [ ]:
# ── 3. Encode categoricals (label encoding for RF) ───────────────────────────
# unknown_int="n_classes" keeps unseen categories non-negative
train_enc, val_enc, test_enc = encode_categoricals(
    train, val, test, method="label", unknown_int="n_classes"
)

In [ ]:
# ── 4. Build feature matrices ────────────────────────────────────────────────
X_train = train_enc[feature_cols].fillna(train_enc[feature_cols].median())
y_train = train_enc[target_cols]

X_val   = val_enc[feature_cols].fillna(train_enc[feature_cols].median())
y_val   = val_enc[target_cols]

X_test  = test_enc[feature_cols].fillna(train_enc[feature_cols].median())
y_test  = test_enc[target_cols]

In [ ]:

# ── 5. Train ─────────────────────────────────────────────────────────────────
rf = RandomForestRegressor(
    n_estimators=200,
    n_jobs=-1,
    random_state=42,
)
rf.fit(X_train, y_train)

In [ ]:
# ── 6. Evaluate ──────────────────────────────────────────────────────────────
for name, X, y in [("val", X_val, y_val), ("test", X_test, y_test)]:
    preds = rf.predict(X)
    r2   = r2_score(y, preds, multioutput="uniform_average")
    rmse = np.sqrt(mean_squared_error(y, preds, multioutput="uniform_average"))
    print(f"{name:5s}  R²={r2:.4f}  RMSE={rmse:.4f}")